# EVALUATION V1 - Weighted Combination



In [1]:
#imports

import pandas as pd
import numpy as np
import random
import ast


from collections import Counter
from ingredient_normalisation import normalise_user_ingredients
from evaluation_queries import make_random_queries, make_manual_queries, MIN_SOURCE_INGREDIENTS
from baseline_retrieval import (tfidf_model, jaccard_model, minilm_model, recipes_dataset_normalised)

# random seeds
split_seed = 34
development_seed = 83

# evaluation settings
# retrieve top 100 candidates before reranking
candidate_k = 100
# compare recommendations at top 5 and top 10
evaluation_k = [5, 10]

# query settings
ingredients_per_query = 3
query_lengths = [3, 4, 5, 6]
development_queries = 50
validation_queries_per_run = 50
test_queries_per_run = 50

# final repeated evaluation
final_runs = 30

recipes_dataset_normalised = pd.read_csv("recipes_dataset_normalised.csv")

# convert stored ingredient strings back into lists
recipes_dataset_normalised["processing_ingredients"] = recipes_dataset_normalised["processing_ingredients"].apply(ast.literal_eval)



c:\Users\preen\Documents\AIML339 - FINAL\AIML339 - FINAL\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# CREATE VALIDATION AND TEST SET

# select recipes with at least 7 different ingredients
valid_recipes = recipes_dataset_normalised[recipes_dataset_normalised["processing_ingredients"].apply(lambda ingredients: len(set(ingredients)) >= MIN_SOURCE_INGREDIENTS)]

# shuffle the source recipe indices using chosen seed 34
recipe_indices = valid_recipes.index.tolist()
random.Random(split_seed).shuffle(recipe_indices)

# split  recipes into validation and test groups
split_point = int(len(recipe_indices) * 0.5)
validation_indices = recipe_indices[:split_point]
test_indices = recipe_indices[split_point:]



In [3]:
# GENERATE DEVELOPMENT QUERIES

# generate the same 50 validation source recipes with 3, 4, 5 and 6 ingredients
# use fixed seeds 34 and 83 - queries are generated again each time
development_query_set = make_random_queries(recipes_dataset_normalised, partition="validation", query_lengths=query_lengths, number_of_queries=development_queries, split_seed=split_seed, query_seed=development_seed, min_source_ingredients=MIN_SOURCE_INGREDIENTS)

# create the 20 fixed manual queries - 5 groups with 3, 4, 5 and 6 ingredients
manual_query_set = make_manual_queries()


In [ ]:
# CHECK RANDOM QUERY GENERATION

# check that 50 source recipes each produce 4 nested queries
assert len(development_query_set) == 200
assert development_query_set["source_index"].nunique() == 50

# check queries belonging to each source recipe together
for source_index, group in development_query_set.groupby("source_index"):
    # order queries from shortest to longest
    group = group.sort_values("query_length")

    # convert query strings back into ingredient lists
    queries = [
        query.split(", ")
        for query in group["query"]
    ]

    # each shorter query must be contained at the start of the longer query
    assert all(
        queries[i] == queries[-1][:len(queries[i])]
        for i in range(len(queries))
    )


All 50 source recipes produced nested queries correctly.


In [5]:
# EVALUATION - RELEVANCE

#assign relevance score based on matched & missing ingredients
def recipe_relevance(user_ingredients, recipe_ingredients):

    # convert ingredients from user and recipe to sets
    user_set = set(user_ingredients)
    recipe_set = set(recipe_ingredients)

    # calculate matched and missing ingredients
    matched = len(user_set & recipe_set)
    missing = len(recipe_set - user_set)

    # no ingredients matched
    if matched == 0:
        return 0

    # some ingredients matched
    if matched < len(user_set):
        return 1

    # all ingredients matched, with few missing
    if missing <= 2:
        return 3

    # all ingredients matched, but more are missing
    return 2

In [6]:
# CCALCULATE RELEVANCE

# relecance for each recipe using user query
def prepare_query_relevance(user_input):
    #normalise ingredients and convert to set
    user_ingredients = normalise_user_ingredients(user_input)
    user_set = set(user_ingredients)

    # relevance score assigned to each recipe - mark recipes containing all ingredients as relevant
    relevance_scores = recipes_dataset_normalised["processing_ingredients"].apply(lambda ingredients: recipe_relevance(user_ingredients, ingredients))
    binary_relevance = recipes_dataset_normalised["processing_ingredients"].apply(lambda ingredients: user_set.issubset(set(ingredients)))
    # total no. of relevant recipes
    total_relevant = int(binary_relevance.sum())

    return relevance_scores, binary_relevance, total_relevant


In [ ]:
#PRECISION, RECALL, NDGC, COVERAGE + MISSING INGREDIENTS

def evaluate_recommendations(recommendations, user_input, k, relevance_scores, binary_relevance, total_relevant):
    # select top k
    top_recipes = recommendations.head(k)
    user_set = set(normalise_user_ingredients(user_input))
    #calculate found relevant in topk
    relevant_found = int(binary_relevance.loc[top_recipes.index].sum())
    # proportion of top k recipes containing all user ingredients
    precision = relevant_found / k
    # proportion of all relevant recipes retrieved - undefined if none are relevant
    recall = relevant_found / total_relevant if total_relevant > 0 else np.nan

    # calculate scores in recommendation orders
    grades = relevance_scores.loc[top_recipes.index].to_numpy()
    #so lower ranked recipes contribute less
    discounts = np.log2(np.arange(2, len(grades) + 2))
    # using relevance scores and rank discounts
    dcg = np.sum((2 ** grades - 1) / discounts)

    # select highest relevance scores in dataset for ideal ranking
    ideal_grades = np.sort(relevance_scores.to_numpy())[::-1][:k]
    ideal_discounts = np.log2(np.arange(2, len(ideal_grades) + 2))
    ideal_dcg = np.sum((2 ** ideal_grades - 1) / ideal_discounts)
    # compare recommendation ranking with ideal ranking
    ndcg = dcg / ideal_dcg if ideal_dcg > 0 else np.nan

    # empty list to add
    coverages = []
    missing_counts = []

    # calculate coverage and missing ingredients for each recommended recipe
    for ingredients in top_recipes["processing_ingredients"]:
        recipe_set = set(ingredients)
        # proportion of user ingredients found in recipe
        coverages.append(len(user_set & recipe_set) / len(user_set))
        # count recipe ingredients user doesnt have
        missing_counts.append(len(recipe_set - user_set))

    return {"precision": precision, "recall": recall, "ndcg": ndcg, "coverage": np.mean(coverages), "missing": np.mean(missing_counts)}


In [8]:
# COMBINED RERANKING V1

# calculate how frequently each ingredient occurs in the recipe dataset
ingredient_counts = Counter()

# go through ingredients of each recipe
for ingredients in recipes_dataset_normalised["processing_ingredients"]:
    # count each ingredient once per recipe
    ingredient_counts.update(set(ingredients))

# proportion of recipes containing each ingredient
ingredient_frequencies = {ingredient: count / len(recipes_dataset_normalised) for ingredient, count in ingredient_counts.items()}

# rerank
def rerank_v1(candidates, user_input, similarity_column):
    # candidate recipes copy to store new reranking scores
    reranked = candidates.copy()
    user_set = set(normalise_user_ingredients(user_input))

    # scale similarity scores between 0 and 1
    scores = reranked[similarity_column]
    #lowest and highest
    minimum = scores.min()
    maximum = scores.max()

    #to avoid dividing by 0
    if maximum == minimum:
        reranked["scaled_similarity"] = 0.0
    else:
        # min-max scaling using lowest and highest candidate similarity
        reranked["scaled_similarity"] = (scores - minimum) / (maximum - minimum)

    # calculate three reranking features 
    def calculate_features(ingredients):
        recipe_set = set(ingredients)
        #ingredients user doesnt have
        missing_set = recipe_set - user_set
        #proportion of user query in recipe
        coverage = len(user_set & recipe_set) / len(user_set)
        # ingredients missing in recipe
        missing_penalty = len(missing_set) / len(recipe_set) if recipe_set else 0
        # reward common missing ingredients
        common_reward = sum(ingredient_frequencies.get(i, 0) for i in missing_set) / len(recipe_set) if recipe_set else 0
        # return three features
        return pd.Series({"coverage": coverage, "missing_penalty": missing_penalty, "common_reward": common_reward})

    # calculate features for each candidate recipe
    features = reranked["processing_ingredients"].apply(calculate_features)
    # add to candidate recipes
    reranked = pd.concat([reranked, features], axis=1)

    # original combined v1 weights
    reranked["final_score"] = 1.0 * reranked["scaled_similarity"] + 0.3 * reranked["coverage"] - 0.2 * reranked["missing_penalty"] + 0.1 * reranked["common_reward"]
    # return in order
    return reranked.sort_values("final_score", ascending=False)


In [9]:
# EVALUATE RANDOM AND MANUAL QUERIES

# store results for each model, map each model to name
evaluation_results = []
models = {"TF-IDF": (tfidf_model, "tfidf_similarity"), "Jaccard": (jaccard_model, "jaccard_similarity"), "MiniLM": (minilm_model, "minilm_similarity")}

# use the same evaluation for random and manual queries
query_sets = {"random": development_query_set, "manual": manual_query_set}

# run random and manual queries through same retrieval models
for experiment, query_set in query_sets.items():
    # go through every query
    for query_number, query_row in query_set.iterrows():
        user_input = query_row["query"]
        # calculate relevance across dataset once for this query
        relevance_scores, binary_relevance, total_relevant = prepare_query_relevance(user_input)

        # go through every retrieval model
        for model_name, (model_function, similarity_column) in models.items():
            # retrieve top 100 recipes using original model similarity
            baseline_candidates = model_function(user_input, candidate_k)
            # reorder same candidates using combined v1 score
            reranked_candidates = rerank_v1(baseline_candidates, user_input, similarity_column)
            configurations = {"baseline": baseline_candidates, "reranked v1": reranked_candidates}
            # evaluate each configuration
            for configuration_name, recommendations in configurations.items():
                # calculate metrics separately for top 5 and top 10
                for k in evaluation_k:
                    metrics = evaluate_recommendations(recommendations, user_input, k, relevance_scores, binary_relevance, total_relevant)
                    evaluation_results.append({"experiment": experiment, "seed": development_seed if experiment == "random" else None, "query_id": query_row["query_id"], "group": query_row.get("group", "random"), "query_length": query_row["query_length"], "query": user_input, "model": model_name, "configuration": configuration_name, "k": k, "total_relevant": total_relevant, **metrics})

# convert results to df
evaluation_results = pd.DataFrame(evaluation_results)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8697.50it/s]


In [10]:
# SUMMARISE DEVELOPMENT EVALUATION

# keep random and manual results separate
random_results = evaluation_results[evaluation_results["experiment"] == "random"].copy()
manual_results = evaluation_results[evaluation_results["experiment"] == "manual"].copy()

# mean and standard deviation across the 50 source queries for each length
random_summary = random_results.groupby(["model", "configuration", "query_length", "k"])[["precision", "recall", "ndcg", "coverage", "missing"]].agg(["mean", "std"]).round(4)
# flatten mean and std column names for display and csv
random_summary.columns = ["_".join(column) for column in random_summary.columns]
# turn grouped model and query settings back into columns
random_summary = random_summary.reset_index()

# results for each fixed manual group
manual_summary = manual_results.groupby(["group", "model", "configuration", "query_length", "k"])[["precision", "recall", "ndcg", "coverage", "missing"]].mean().round(4).reset_index()

# retain the original combined results and summary
evaluation_summary = random_summary

display(random_summary)
display(manual_summary)


,model,configuration,query_length,k,precision_mean,precision_std,recall_mean,recall_std,ndcg_mean,ndcg_std,coverage_mean,coverage_std,missing_mean,missing_std
0,Jaccard,baseline,3,5,0.304,0.2563,0.4212,0.4256,0.7383,0.2107,0.7160,0.1339,2.748,1.0735
1,Jaccard,baseline,3,10,0.238,0.2355,0.5125,0.4273,0.7778,0.1839,0.6913,0.1386,3.126,1.1482
2,Jaccard,baseline,4,5,0.292,0.2329,0.7012,0.4013,0.8633,0.1824,0.7240,0.1341,3.284,1.3335
3,Jaccard,baseline,4,10,0.198,0.2025,0.7588,0.3645,0.8822,0.1537,0.6750,0.1492,3.480,1.1636
4,Jaccard,baseline,5,5,0.260,0.1471,0.8900,0.2877,0.9495,0.1234,0.7024,0.1360,3.412,1.3064
5,Jaccard,baseline,5,10,0.154,0.1631,0.9018,0.2770,0.9560,0.1153,0.6492,0.1490,3.796,1.3018
6,Jaccard,baseline,6,5,0.252,0.1752,0.9230,0.2479,0.9809,0.0790,0.6813,0.1342,3.308,1.2957
7,Jaccard,baseline,6,10,0.150,0.1741,0.9525,0.1881,0.9881,0.0438,0.6180,0.1438,3.670,1.3314
8,Jaccard,reranked v1,3,5,0.348,0.2823,0.4687,0.4189,0.7662,0.2017,0.7360,0.1406,2.984,1.2725
9,Jaccard,reranked v1,3,10,0.278,0.2493,0.5787,0.4049,0.8051,0.1665,0.7127,0.1393,3.380,1.2155


,group,model,configuration,query_length,k,precision,recall,ndcg,coverage,missing
0,beef_rice,Jaccard,baseline,3,5,0.4,0.1053,0.5182,0.8000,3.0
1,beef_rice,Jaccard,baseline,3,10,0.3,0.1579,0.4957,0.7667,3.3
2,beef_rice,Jaccard,baseline,4,5,0.4,0.1250,0.5182,0.8000,2.8
3,beef_rice,Jaccard,baseline,4,10,0.3,0.1875,0.4957,0.8000,3.2
4,beef_rice,Jaccard,baseline,5,5,0.6,0.3000,0.7443,0.8800,3.6
...,...,...,...,...,...,...,...,...,...,...
235,white_cake,TF-IDF,reranked v1,4,10,0.5,0.0139,0.4600,0.8750,1.1
236,white_cake,TF-IDF,reranked v1,5,5,0.4,0.0160,0.4387,0.8800,0.8
237,white_cake,TF-IDF,reranked v1,5,10,0.4,0.0320,0.4649,0.8800,1.1
238,white_cake,TF-IDF,reranked v1,6,5,0.2,0.0167,0.4336,0.8667,0.4


In [11]:
# SAVE DEVELOPMENT EVALUATION RESULTS

# save the random and manual results separately
random_results.to_csv("evaluation1_randomquery.csv", index=False)
random_summary.to_csv("evaluation1_randomquery_summary.csv", index=False)
manual_results.to_csv("evaluation1_manual_userqueries.csv", index=False)
manual_summary.to_csv("evaluation1_manual_userqueries_summary.csv", index=False)

# keep the original development filenames as well
evaluation_results.to_csv("evaluation1_development_results.csv", index=False)
evaluation_summary.to_csv("evaluation1_development_summary.csv", index=False)
